# NB5: Paper-Scale Evaluation (PBSGAN Comparison)

**Paper:** Xu et al., "Aspect category sentiment analysis based on pre-trained BiLSTM and syntax-aware graph attention network", Scientific Reports 2025.

**Metric:** ACSA Accuracy (%) with gold aspect categories.

**Datasets:**
- **D14** -- SemEval 2014 Task 4 Restaurant (973 test opinions)
- **D14-hard** -- Subset: sentences with >=2 categories AND >=2 distinct polarities
- **MAMS** -- Multi-Aspect Multi-Sentiment (760 test opinions)

**Models evaluated:**
- No-Retrieval (DeBERTa baseline)
- Retrieval + Aux Loss (Label Interpolation)

**Embedding:** polonly + MAMS (`embedding_2014_mams_s2`) for both datasets.

**Note:** Paper splits D14 at 8:1:1; we use the official SemEval train/test split (3044/800 sentences).

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os, sys, json, shutil, re, subprocess
from collections import defaultdict

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Wire Data + Checkpoints

In [ ]:
# ---- Helper: find Kaggle dataset path ----
def find_dataset(name, owners=['duclm318', 'lcminhc']):
    candidates = [f'/kaggle/input/{name}']
    for owner in owners:
        candidates.append(f'/kaggle/input/datasets/{owner}/{name}')
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f'Dataset {name} not found. Tried: {candidates}')

# ---- SemEval 2014 ----
SEM_DS = find_dataset('semeval-2014-absa-restaurant')
os.makedirs('SemEval-2014', exist_ok=True)
shutil.copy(f'{SEM_DS}/Restaurants_Train.xml', 'SemEval-2014/Restaurants_Train.xml')
shutil.copy(f'{SEM_DS}/Restaurants_Test_Gold.xml', 'SemEval-2014/Restaurants_Test_Gold.xml')
!python scripts/01_prepare_data.py
print('SemEval 2014 data prepared.')

# ---- MAMS ----
!git clone https://github.com/siat-nlp/MAMS-for-ABSA.git data/mams 2>/dev/null || echo 'MAMS already cloned'
!python scripts/01_prepare_data_mams.py
print('MAMS data prepared.')

# ---- Embedding checkpoint ----
EMB_DS = find_dataset('p5-embed-v4')
print(f'Embedding dataset: {EMB_DS}')
os.listdir(EMB_DS)

In [ ]:
# ---- Wire embedding ----
EMB_CKPT_NAME = 'embedding_v4_s2_best.pt'  # adjust if filename differs
os.makedirs('checkpoints/embedding', exist_ok=True)
shutil.copy(f'{EMB_DS}/{EMB_CKPT_NAME}', 'checkpoints/embedding/best.pt')
EMB_CKPT = 'checkpoints/embedding/best.pt'
print(f'Embedding: {os.path.getsize(EMB_CKPT) / 1e6:.1f} MB')

# ---- Wire Stage 2 checkpoints ----
# Adjust dataset name(s) to match your Kaggle uploads
STAGE2_DS = find_dataset('p5-nb5-stage2-checkpoints')  # <-- CHANGE THIS

CKPT_MAP = {
    'stage2_2014_noret':   'stage2_2014_noret_best.pt',
    'stage2_2014_auxloss': 'stage2_2014_auxloss_best.pt',
    'stage2_mams_noret':   'stage2_mams_noret_best.pt',
    'stage2_mams_auxloss': 'stage2_mams_auxloss_best.pt',
}

for ckpt_dir, filename in CKPT_MAP.items():
    os.makedirs(f'checkpoints/{ckpt_dir}', exist_ok=True)
    src = f'{STAGE2_DS}/{filename}'
    dst = f'checkpoints/{ckpt_dir}/best.pt'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'{ckpt_dir}: {os.path.getsize(dst)/1e6:.1f} MB')
    else:
        print(f'WARNING: {src} not found')

print('\nAll checkpoints wired.')

## 2. Build FAISS Indexes

In [ ]:
os.makedirs('indexes', exist_ok=True)
os.makedirs('indexes/mams', exist_ok=True)

print('--- Building SemEval FAISS index ---')
!python scripts/03_build_index.py \
    --embedding_ckpt {EMB_CKPT} \
    --input data/processed/classification.jsonl \
    --out_dir indexes/

print('\n--- Building MAMS FAISS index ---')
!python scripts/03_build_index.py \
    --embedding_ckpt {EMB_CKPT} \
    --input data/processed_mams/classification.jsonl \
    --out_dir indexes/mams/

## 3. Create D14-hard Subset

In [ ]:
from src.utils.io import read_jsonl

records = read_jsonl('data/processed/sentiment_records.jsonl')
test_records = [r for r in records if r['split'] == 'test']

sent_groups = defaultdict(list)
for r in test_records:
    sid = r['id'].rsplit('_', 1)[0]
    sent_groups[sid].append(r)

hard_records = []
for sid, recs in sent_groups.items():
    if len(recs) >= 2:
        polarities = set(r['polarity'] for r in recs)
        if len(polarities) >= 2:
            hard_records.extend(recs)

os.makedirs('data/processed_d14hard', exist_ok=True)
with open('data/processed_d14hard/sentiment_records.jsonl', 'w') as f:
    for r in hard_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

hard_sents = len(set(r['id'].rsplit('_', 1)[0] for r in hard_records))
print(f'D14-hard: {hard_sents} sentences, {len(hard_records)} records')
print(f'(out of {len(set(r["id"].rsplit("_", 1)[0] for r in test_records))} total test sentences)')

## 4. Evaluate All Combinations

In [ ]:
EXPERIMENTS = [
    {"name": "D14 / No-Ret",     "data_dir": "data/processed",        "ckpt_dir": "checkpoints/stage2_2014_noret",   "config": "configs/stage2_2014_noret.yaml",   "no_ret": True,  "index_dir": None},
    {"name": "D14 / Ret+Aux",    "data_dir": "data/processed",        "ckpt_dir": "checkpoints/stage2_2014_auxloss", "config": "configs/stage2_2014_auxloss.yaml", "no_ret": False, "index_dir": "indexes/"},
    {"name": "D14h / No-Ret",    "data_dir": "data/processed_d14hard", "ckpt_dir": "checkpoints/stage2_2014_noret",   "config": "configs/stage2_2014_noret.yaml",   "no_ret": True,  "index_dir": None},
    {"name": "D14h / Ret+Aux",   "data_dir": "data/processed_d14hard", "ckpt_dir": "checkpoints/stage2_2014_auxloss", "config": "configs/stage2_2014_auxloss.yaml", "no_ret": False, "index_dir": "indexes/"},
    {"name": "MAMS / No-Ret",    "data_dir": "data/processed_mams",   "ckpt_dir": "checkpoints/stage2_mams_noret",   "config": "configs/stage2_mams_noret.yaml",   "no_ret": True,  "index_dir": None},
    {"name": "MAMS / Ret+Aux",   "data_dir": "data/processed_mams",   "ckpt_dir": "checkpoints/stage2_mams_auxloss", "config": "configs/stage2_mams_auxloss.yaml", "no_ret": False, "index_dir": "indexes/mams/"},
]

def parse_eval_output(stdout):
    result = {}
    for line in stdout.split('\n'):
        if 'Accuracy:' in line:
            m = re.search(r'Accuracy:\s+([\d.]+)', line)
            if m: result['acc'] = float(m.group(1))
        elif 'Macro F1:' in line:
            m = re.search(r'Macro F1:\s+([\d.]+)', line)
            if m: result['macro_f1'] = float(m.group(1))
        elif 'Per-polarity:' in line:
            m = re.search(r'pos=([\d.]+)\s+neg=([\d.]+)\s+neu=([\d.]+)', line)
            if m:
                result['f1_pos'] = float(m.group(1))
                result['f1_neg'] = float(m.group(2))
                result['f1_neu'] = float(m.group(3))
        elif 'Total opinions:' in line:
            m = re.search(r'Total opinions:\s+(\d+)', line)
            if m: result['n'] = int(m.group(1))
    return result

results = {}
for exp in EXPERIMENTS:
    ckpt = f'{exp["ckpt_dir"]}/best.pt'
    if not os.path.exists(ckpt):
        print(f'SKIP {exp["name"]} -- checkpoint missing: {ckpt}')
        continue

    cmd = ["python", "scripts/06_evaluate_sentiment_only.py",
           "--stage2_ckpt", ckpt,
           "--stage2_config", exp["config"],
           "--data_dir", exp["data_dir"]]

    if exp["no_ret"]:
        cmd.append("--no_retrieval")
    else:
        cmd += ["--embedding_ckpt", EMB_CKPT,
                "--index_dir", exp["index_dir"]]

    print(f'\n{"="*60}')
    print(f'Running: {exp["name"]}')
    print(f'{"="*60}')
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(f'STDERR: {proc.stderr[-500:]}')

    parsed = parse_eval_output(proc.stdout)
    if parsed:
        results[exp['name']] = parsed
        print(f'  -> Acc={parsed.get("acc",0)*100:.2f}%  MacF1={parsed.get("macro_f1",0):.4f}  (n={parsed.get("n","?")})')
    else:
        print(f'  -> PARSE FAILED')

print(f'\n{"="*60}')
print(f'Collected {len(results)}/{len(EXPERIMENTS)} results')

## 5. Comparison Table

In [ ]:
# ---- Paper baselines (Table 3, Xu et al. 2025) ----
PAPER_BASELINES = [
    {"method": "LSTM",       "D14": 80.90, "D14h": 47.93, "MAMS": 46.61},
    {"method": "BiLSTMAttn", "D14": 82.15, "D14h": 50.18, "MAMS": 49.06},
    {"method": "AT-LSTM",    "D14": 82.67, "D14h": 54.72, "MAMS": 66.44},
    {"method": "ATAE-LSTM",  "D14": 82.14, "D14h": 56.61, "MAMS": 70.63},
    {"method": "GCAE",       "D14": 81.33, "D14h": 54.72, "MAMS": 72.10},
    {"method": "CapsNet",    "D14": 81.17, "D14h": 53.96, "MAMS": 73.99},
    {"method": "AC-MIMLLN",  "D14": 81.60, "D14h": 65.28, "MAMS": 76.43},
    {"method": "M-AT-LSTM",  "D14": 81.28, "D14h": 60.76, "MAMS": 74.65},
    {"method": "PBSGAN",     "D14": 81.91, "D14h": 68.52, "MAMS": 74.57},
]

def get_acc(name):
    r = results.get(name)
    return f'{r["acc"]*100:.2f}' if r else '--'

# ---- Table 1: ACSA Accuracy comparison ----
print('=' * 72)
print('TABLE 1: ACSA Accuracy (%) -- Comparison with PBSGAN (2025)')
print('=' * 72)
print(f'{"Method":<20} {"D14":>8} {"D14-hard":>10} {"MAMS":>8}   Source')
print('-' * 72)
for b in PAPER_BASELINES:
    print(f'{b["method"]:<20} {b["D14"]:>8.2f} {b["D14h"]:>10.2f} {b["MAMS"]:>8.2f}   PBSGAN 2025')
print('-' * 72)
print(f'{"Ours (No-Ret)":<20} {get_acc("D14 / No-Ret"):>8} {get_acc("D14h / No-Ret"):>10} {get_acc("MAMS / No-Ret"):>8}   This work')
print(f'{"Ours (Ret+Aux)":<20} {get_acc("D14 / Ret+Aux"):>8} {get_acc("D14h / Ret+Aux"):>10} {get_acc("MAMS / Ret+Aux"):>8}   This work')
print('=' * 72)
print()
print('Note: Paper splits D14 at 8:1:1. We use official SemEval train/test split.')
print(f'      D14-hard = {results.get("D14h / No-Ret", {}).get("n", "?")} test opinions '
      f'(sentences with >=2 categories + >=2 distinct polarities).')

In [ ]:
# ---- Table 2: Detailed per-polarity breakdown ----
print('=' * 80)
print('TABLE 2: Detailed Metrics (our models only)')
print('=' * 80)
print(f'{"Dataset":<10} {"Strategy":<12} {"N":>5} {"Acc":>8} {"MacF1":>8} {"pos":>8} {"neg":>8} {"neu":>8}')
print('-' * 80)

ROW_ORDER = [
    ('D14',      'No-Ret',   'D14 / No-Ret'),
    ('D14',      'Ret+Aux',  'D14 / Ret+Aux'),
    ('D14-hard', 'No-Ret',   'D14h / No-Ret'),
    ('D14-hard', 'Ret+Aux',  'D14h / Ret+Aux'),
    ('MAMS',     'No-Ret',   'MAMS / No-Ret'),
    ('MAMS',     'Ret+Aux',  'MAMS / Ret+Aux'),
]

for ds, strat, key in ROW_ORDER:
    r = results.get(key)
    if r:
        print(f'{ds:<10} {strat:<12} {r.get("n","?"):>5} '
              f'{r["acc"]*100:>7.2f}% {r.get("macro_f1",0):>8.4f} '
              f'{r.get("f1_pos",0):>8.4f} {r.get("f1_neg",0):>8.4f} {r.get("f1_neu",0):>8.4f}')
    else:
        print(f'{ds:<10} {strat:<12}   --       --       --       --       --       --')

print('=' * 80)

## 6. Save Results

In [ ]:
OUT_DIR = '/kaggle/working/outputs_p5_nb5'
os.makedirs(OUT_DIR, exist_ok=True)

# ---- JSON summary ----
summary = {
    'paper': 'PBSGAN (Xu et al., Scientific Reports 2025)',
    'metric': 'ACSA Accuracy (gold categories)',
    'note': 'Paper splits D14 at 8:1:1; we use official SemEval train/test split.',
    'paper_baselines': PAPER_BASELINES,
    'our_results': {k: v for k, v in results.items()},
}
with open(f'{OUT_DIR}/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# ---- Markdown report ----
md_lines = [
    '# ACSA Evaluation -- PBSGAN Comparison',
    '',
    '## Accuracy (%)',
    '',
    f'| Method | D14 | D14-hard | MAMS | Source |',
    f'|--------|-----|----------|------|--------|',
]
for b in PAPER_BASELINES:
    md_lines.append(f'| {b["method"]} | {b["D14"]:.2f} | {b["D14h"]:.2f} | {b["MAMS"]:.2f} | PBSGAN 2025 |')
md_lines.append(f'| **Ours (No-Ret)** | **{get_acc("D14 / No-Ret")}** | **{get_acc("D14h / No-Ret")}** | **{get_acc("MAMS / No-Ret")}** | This work |')
md_lines.append(f'| **Ours (Ret+Aux)** | **{get_acc("D14 / Ret+Aux")}** | **{get_acc("D14h / Ret+Aux")}** | **{get_acc("MAMS / Ret+Aux")}** | This work |')

md_lines += [
    '',
    '## Detailed Metrics',
    '',
    '| Dataset | Strategy | N | Acc (%) | Macro F1 | pos F1 | neg F1 | neu F1 |',
    '|---------|----------|---|--------|----------|--------|--------|--------|',
]
for ds, strat, key in ROW_ORDER:
    r = results.get(key)
    if r:
        md_lines.append(
            f'| {ds} | {strat} | {r.get("n","?")} | {r["acc"]*100:.2f} | '
            f'{r.get("macro_f1",0):.4f} | {r.get("f1_pos",0):.4f} | '
            f'{r.get("f1_neg",0):.4f} | {r.get("f1_neu",0):.4f} |')

md_lines += ['', f'Note: Paper splits D14 at 8:1:1; we use official SemEval train/test split.', '']

with open(f'{OUT_DIR}/paper_comparison.md', 'w') as f:
    f.write('\n'.join(md_lines))

print(f'Results saved to {OUT_DIR}/')
print(os.listdir(OUT_DIR))